In [ ]:
"""
코랩용 DPO (Direct Preference Optimization) 학습 스크립트
./finetuning_data_dpo의 cycle_01.csv 파일을 토대로 1 사이클 DPO 학습 이후
./checkpoints_dpo에 Trainer 등의 메타 데이터를 저장하고 이후 resume을 통해 추가 학습할 수 있도록 함.
adapter의 경우 /content/drive/Mydrive/멋사/adapters_dpo_1_v2/에 저장
"""

In [1]:
import torch
torch.cuda.is_available()

True

In [2]:
!pip install datasets peft trl bitsandbytes accelerate
!pip install -U transformers
!pip show transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.5/532.5 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 14.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 131.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.3
    Uninstalling transformers-4.57.3:
      Successfully uninstalled transformers-4.57.3
Name: transformers
Version: 4.57.6
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, r

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
print(os.getcwd())
print(os.listdir())

/content
['.config', 'drive', '.env', '.ipynb_checkpoints', 'sample_data']


In [5]:
!git clone https://github.com/jjjh02/AmoRe_crm_generator.git
%cd AmoRe_crm_generator
!git checkout jinhyeok
!git branch
os.chdir("/content/AmoRe_crm_generator")
print(os.getcwd())

Cloning into 'AmoRe_crm_generator'...
remote: Enumerating objects: 426, done.
remote: Counting objects: 100% (164/164), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 426 (delta 82), reused 97 (delta 41), pack-reused 262 (from 1)
Receiving objects: 100% (426/426), 4.85 MiB | 7.42 MiB/s, done.
Resolving deltas: 100% (237/237), done.
/content/AmoRe_crm_generator
Branch 'jinhyeok' set up to track remote branch 'jinhyeok' from 'origin'.
Switched to a new branch 'jinhyeok'
* jinhyeok
  main
/content/AmoRe_crm_generator


In [6]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
import os
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
from datasets import load_dataset
from peft import LoraConfig, PeftModel
from trl import DPOTrainer, DPOConfig

# 모델 및 경로 설정
MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"
CACHE_DIR = "./models"
OUTPUT_DIR = "./finetuning/checkpoints_dpo"
OUTPUT_ADAPTER_DIR = "/content/drive/MyDrive/LikeLion/adapters_dpo_1_v4"
BASE_ADAPTER_PATH = "/content/drive/MyDrive/LikeLion/adapters_sft_1_v2"
NEW_ADAPTER_NAME = "dpo_adapter_v4"

# 데이터셋 경로 설정
DATA_DIR = "/content/AmoRe_crm_generator/finetuning/finetuning_data/crm-dpo-dataset"
JSON_FILE = os.path.join(DATA_DIR, "cycle_01_v4.jsonl")

# 하이퍼파라미터 설정
PROMPT_LENGTH = 1024
MAX_SEQ_LENGTH = 1512


def load_dpo_dataset(json_path: str):
    """JSON 파일에서 DPO 형식의 데이터셋을 로드합니다.

    JSON 형식:
    [
      { "prompt": "...", "chosen": "...", "rejected": "..." },
      ...
    ]

    Args:
        json_path: JSON 파일 경로

    Returns:
        train_dataset, eval_dataset
    """
    # JSON 파일 로드
    dataset = load_dataset(
        "json",
        data_files=json_path,
    )
    dataset = dataset["train"]

    # train / eval split
    dataset = dataset.train_test_split(test_size=0.1, seed=42)

    return dataset["train"], dataset["test"]


def _freeze_all_params(model):
    for _, param in model.named_parameters():
        param.requires_grad = False


def _enable_adapter_params(model, adapter_name):
    for name, param in model.named_parameters():
        if f".{adapter_name}." in name:
            param.requires_grad = True


In [8]:
"DPO 학습 메인 함수"

# 1. 토크나이저 로드
print("토크나이저 로드 중...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    cache_dir=CACHE_DIR,
)

# pad_token 설정
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 패딩 사이드 설정 (DPO 학습에 유리)
tokenizer.padding_side = 'left'
tokenizer.truncation_side = 'left'

# max_length 설정
tokenizer.model_max_length = MAX_SEQ_LENGTH

# 2. 데이터셋 로드
print(f"데이터셋 로드 중: {JSON_FILE}")
if not os.path.exists(JSON_FILE):
    raise FileNotFoundError(f"데이터셋 파일을 찾을 수 없습니다: {JSON_FILE}")

train_dataset, eval_dataset = load_dpo_dataset(JSON_FILE)
print(f"학습 데이터: {len(train_dataset)}개, 평가 데이터: {len(eval_dataset)}개")

# 3. Flash Attention 설정
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    attn_implementation = "flash_attention_2"
    torch_dtype = torch.bfloat16
else:
    attn_implementation = "eager"
    torch_dtype = torch.float16

# 4. 모델 로드
print("모델 로드 중...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    use_cache=False,
    # attn_implementation=attn_implementation,
    torch_dtype=torch_dtype,
    cache_dir=CACHE_DIR,
)

# 5. PEFT (LoRA) 설정
print("PEFT 설정 중...")
peft_config = LoraConfig(
    lora_alpha=64,
    lora_dropout=0.05,
    r=64,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    task_type="CAUSAL_LM"
)

# 6. 베이스 어댑터 로드 (SFT한 어댑터)
print(f"베이스 어댑터 로드 중: {BASE_ADAPTER_PATH}")
if not os.path.exists(BASE_ADAPTER_PATH):
    raise FileNotFoundError(f"베이스 어댑터를 찾을 수 없습니다: {BASE_ADAPTER_PATH}")

model = PeftModel.from_pretrained(
    model,
    BASE_ADAPTER_PATH,
    is_trainable=True,
)
model.print_trainable_parameters()

# 7. 추가 어댑터 생성 및 활성화
# print(f"추가 어댑터 생성: {NEW_ADAPTER_NAME}")
# model.add_adapter(peft_config, NEW_ADAPTER_NAME)
# model.set_adapter(NEW_ADAPTER_NAME)
# _freeze_all_params(model)
# _enable_adapter_params(model, NEW_ADAPTER_NAME)

# 8. DPO Config 설정
print("DPO Config 설정 중...")
dpo_config = DPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=3,
    learning_rate=1e-5,
    max_grad_norm=0.3,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=1,
    logging_first_step=True,
    logging_strategy="steps",
    log_level="info",
    disable_tqdm=False,
    save_steps=100,
    save_total_limit=20,
    eval_strategy="steps",
    eval_steps=10,
    # fp16=True,
    beta=0.3,
    loss_type="sigmoid",
    report_to="none"
)

# 9. DPOTrainer 초기화
print("DPOTrainer 초기화 중...")
trainer = DPOTrainer(
    model=model,
    ref_model=None,  # PEFT 사용 시 None으로 설정
    args=dpo_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

# 10. 학습 시작
print("학습 시작...")
ckpt_dir = "AmoRe_crm_generator/finetuning/checkpoints_dpo"

resume = None
if os.path.isdir(ckpt_dir) and len(os.listdir(ckpt_dir)) > 0:
    resume = True

trainer.train(resume_from_checkpoint=resume)

# 11. 모델 저장
print("모델 저장 중...")
trainer.save_model(OUTPUT_ADAPTER_DIR)
print(f"모델이 저장되었습니다: {OUTPUT_ADAPTER_DIR}")



토크나이저 로드 중...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
10,1.222600,1.782604,10.163637,10.669087,0.454545,-0.505451,-141.069656,-163.528595,-4.052771,-3.842360
20,0.683000,0.871584,9.248056,7.430384,0.704545,1.817672,-144.121597,-174.324295,-4.284035,-4.065804
30,0.335400,0.487282,6.785668,2.299005,0.795455,4.486664,-152.329559,-191.428909,-4.662226,-4.444969
40,0.075400,0.313066,5.515716,-0.739620,0.871212,6.255335,-156.562744,-201.557632,-4.965423,-4.754317
50,0.356900,0.260306,4.841506,-2.628740,0.916667,7.470246,-158.810089,-207.854706,-5.204030,-4.984785
60,0.287100,0.280865,3.884565,-4.052152,0.924242,7.936718,-161.999893,-212.599396,-5.577482,-5.355666
70,0.647700,0.258894,2.443793,-5.630293,0.954545,8.074085,-166.802475,-217.859879,-6.023201,-5.779470
80,0.005800,0.196153,1.975271,-6.386849,0.969697,8.362120,-168.364227,-220.381714,-6.515690,-6.276878
90,0.101900,0.199811,0.832484,-8.105106,0.977273,8.937591,-172.173508,-226.109253,-6.935230,-6.677631
100,0.017600,0.182622,-0.079438,-9.296011,0.977273,9.216575,-175.213242,-230.078934,-7.354593,-7.091055


The following columns in the Evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: prompt, sent_idx, violation_type, line_idx, violation_any. If prompt, sent_idx, violation_type, line_idx, violation_any are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 130
  Batch size = 4
The following columns in the Evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: prompt, sent_idx, violation_type, line_idx, violation_any. If prompt, sent_idx, violation_type, line_idx, violation_any are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 130
  Batch size = 4
The following columns in the Evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: prompt, sent_idx, violation_typ

config.json: 0.00B [00:00, ?B/s]

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--LGAI-EXAONE--EXAONE-4.0-1.2B/snapshots/3abf2810673c7c0778df64a73c2d52eab32d91c4/config.json
Model config Exaone4Config {
  "architectures": [
    "Exaone4ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "dtype": "bfloat16",
  "eos_token_id": 361,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attenti

KeyboardInterrupt: 

In [ ]:
!pip install huggingface-hub

In [ ]:
# Push to HuggingFace Hub

import os

from dotenv import load_dotenv
from huggingface_hub import login, create_repo, upload_folder

login(os.getenv("HUGGINGFACE_API_KEY"))

create_repo(
    repo_id="crm-dpo-adapter",
    repo_type="model",
    private=False,
    exist_ok=True
)

upload_folder(
    folder_path=OUTPUT_ADAPTER_DIR,
    repo_id="jinn33/crm-dpo-adapter",
    repo_type="model",
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   1%|          |  620kB /  122MB            

  ...po_1_v2/training_args.bin:   1%|1         |  76.0B / 6.76kB            

CommitInfo(commit_url='https://huggingface.co/jinn33/crm-dpo-adapter/commit/38e95322898190e4a5295f408a79a138ae55ca16', commit_message='Upload folder using huggingface_hub', commit_description='', oid='38e95322898190e4a5295f408a79a138ae55ca16', pr_url=None, repo_url=RepoUrl('https://huggingface.co/jinn33/crm-dpo-adapter', endpoint='https://huggingface.co', repo_type='model', repo_id='jinn33/crm-dpo-adapter'), pr_revision=None, pr_num=None)